<div align="right"><i>Matías Torres Esteban<br>Enero, 2026</i></div>

# Generación Aumentada por Recuperación

En esta notebook mostramos las herramientas provistas por nuestra librería para crear Sistemas de Generación Aumentada por Recuperación (RAG) [1], los cuales integran grandes modelos de lenguaje (LLMs) con técnicas de recuperación de información (RI) para responder preguntas de dominios expertos. Nuestra librería implementa dos estrategias de RAG:  

1. **Naive RAG:** Recupera los $k$ chunks más relevantes a una consulta escrita por el usuario en lenguaje natural. El LLM es instruido a responder basándose exlusivamente en la información provista en los chunks.
2. **CMap RAG:** Recupera los $k$ mapas conceptuales más relevantes a una consulta escrita por el usuario. El LLM responde basándose exclusivamente en las ternas de conocimiento de los mapas conceptuales.

Nuestra intención principal es mostrar estrategias simples pero eficaces para fundamentar las respuestas de LLMs grandes y pequeños en nuestra base de conocimiento. Muchas modificaciones y extensiones pueden realizarse a los algoritmos que mostramos aquí, pero estos dependen de las características particulares del dominio donde se aplican y de las necesidades especiales del usuario. Sin embargo, son pocos los conceptos fundamentales que hay que aprender para desarrolar sistemas inteligentes robustos, interpretables y confiables, y analizar una implementación real de estos conceptos puede ser una gran ayuda para crear sistemas propios. 

Aplicamos nuestros algoritmos RAG para responder preguntas referidas a la salud mental y la anatomía de cerebro. Vamos a trabajar sobre una base de conocimiento que almacena entradas del diccionario de la APA [2] y los mapas conceptuales obtenidos a partir de estas entradas. 

## 1. Naive RAG

El módulo `chunk_qa` contiene las componentes necesarias para implementar un algoritmo RAG clásico. La lógica principal es llevada a cabo por la clase `ChunkQALogic`, la cual actúa como interfaz entre la base de conocimiento y los usuarios del sistema. Esta es responsable de la validación de los datos de entrada, el control de errores y la sincronización entre el módulo de persistencia y los objetos que interactúan con LLMs. Su punto de entrada es el método `answer_query`, el cual recibe un string *query* con la consulta del usuario y un entero $k$ que especifica el número de chunks a recuperar antes de generar una respuesta. Su flujo de trabajo es el siguiente:

1. **Validación:** Se controla que la consulta *query* y el número de chunks a recuperar $k$ sean valores correctos - Por ejemplo, que $k$ sea un entero no negativo y *query* no sea una cadena vacía.
2. **Codificación:** Se calcula el embedding de la consulta *query* mediante un transformador de sentencias.
3. **Recuperación:** Se invoca al módulo de persistencia para realizar una búsqueda semántica de los $k$ chunks más relevantes a *query*.
4. **Generación:** Se invoca a un objeto del tipo `Generator` y que interactúa con un LLM para responder *query* en función de los chunks recuperados. Este genera un objeto del tipo `LLMInvocationData` con la respuesta en crudo del modelo. 
5. **Extracción:** Se invoca a un objeto del tipo `Generator` para analizar sintácticamente la respuesta del generador y estructurar su contenido como un objeto del tipo `Answer`. El análisis sintáctico lo realiza un LLM que se configura de manera especial para que sus respuestas cumplan fielmente una gramática preestablecida.

El resultado de todo este proceso es un objeto del tipo `ChunkQAResult` y que contiene los resultados de la validación, los chunks recuperados de la base semántica, las invocaciones del generador y el extractor y una respuesta estructurada a la consulta. Esta última consiste de una lista de frases extraidas de los chunks y que son relevantes a la pregunta de entrada junto a una conclusión readactada a partir de estas frases. Todos estos elementos permiten al usuario interpretar y validar la respuesta del modelo y habilitan al programador a diagnosticar el funcionamiento del proceso. 

Importamos continuación las librerías necesarias, establecemos la conexión con la base de conocimiento y creamos los objetos que nos permiten implementar el sistema RAG. 


In [1]:
from llms_kgs.logic.chunk_qa import (
    # Prompts:
    GENERATOR_ONE_SHOT_SYSTEM_PROMPT,
    EXTRACTOR_SYSTEM_TEMPLATE,
    GENERATOR_USER_TEMPLATE,

    # Generator components:
    PromptFormatter, 
    Generator,

    # Extractor components:
    Answer,
    Extractor,

    # Orchestrator:
    ChunkQALogic,
    ChunkQAResult)

from llms_kgs.persistence import (
    ConnectionProvider,
    ChunkRepository)

from llms_kgs.domain import Chunk, Query

from llms_kgs.llms import Gemma3_4B, M3Encoder, Gemini25Flash

# Establish connection with knowledge base:
connection_provider = ConnectionProvider(
    dbname='nsym',host='localhost',user='postgres',password='postgres',port='5432')

# Instantiate chunk retriever: 
retriever = ChunkRepository(connection_provider)

# Instantiate sentence transformer:
m3 = M3Encoder()

# Instantiate generator prompt formatter:
formatter = PromptFormatter(GENERATOR_USER_TEMPLATE)

# Instantiate generator language model: 
generator_llm = Gemini25Flash()

# Instantiate generator: 
generator = Generator(system_prompt = GENERATOR_ONE_SHOT_SYSTEM_PROMPT,
    formatter = formatter, llm = generator_llm)

# Instantiate extractor: 
extractor_llm = Gemma3_4B(schema=Answer.model_json_schema())
extractor = Extractor(extractor_llm)

# Instantiate logic component: 
chunk_qa = ChunkQALogic(retriever = retriever, extractor = extractor,
                        generator = generator, ef = m3)

Vemos a partir del código de construcción de objetos que la obtención de una respuesta por parte de un LLM está basado en una arquitectura Generación - Extracción. La componente de generación instruye al modelo a razonar y generar una respuesta a la consulta del usuario pero no impone restricciones fuertes respecto al formato de sus salidas. Por otro lado, la componente de extracción comanda a un LLM a que estructure la respuesta del generador en un formato estricto - Especificamente, como un objeto del tipo `Answer` - pero no realiza ningún tipo de razonamiento sobre la consulta. La intuición detrás de esta arquitectura es que ambas tareas - razonamiento sobre una pregunta y formateo de una respuesta - deben realizarse en dos invocaciones separadas porque son de naturaleza muy distinta. Los LLMs pequeños se *sobrecargan* de responsabilidades al solicitarles realizar ambas tareas en una misma invocación y no son efectivos en una ni en otra.

A continuación probamos el funcionamiento del generador sobre textos inventados por nosotros. Este interactúa con el modelo Gemma3:4B para extraer frases de los chunks que ayuden a responder la consulta y a generar una respuesta final. Imprimimos tanto la respuesta del modelo como los prompts de usuario y sistema utilizados.  

In [2]:
chunks = [
    
Chunk(text="""Emotional intelligence is the capacity to cope with life's demands and control \
one's inner experience to maximize effectiveness and well-being. Achieving it requires practicing \
virtues and learning to recognize both personal and collective emotions."""),
    
Chunk(text="""Good mental health can be recognized by a person's smile and posture."""),
    
Chunk(text="""Aristoteles was a philosopher who argued that happiness is achieved by practicing \
and acquiring virtues.""")]

query = Query('How we may achieve happiness?')

generator_invocation = generator.generate(query, chunks)

print("<LLM Answer>")
print(generator_invocation.raw_answer)
print("="*80)
print("<System Prompt>")
print(generator_invocation.system_prompt)
print("="*80)
print("<User Prompt>")
print(generator_invocation.user_prompt)

<LLM Answer>
Passage: [Chunk 2] Aristoteles was a philosopher who argued that happiness is achieved by practicing and acquiring virtues.

Final Answer:
Happiness is achieved by practicing and acquiring virtues.
<System Prompt>
You are an expert in psychology. Your task is to answer user queries ONLY using the information contained in the provided chunks.

Each chunk is prefixed with a unique identifier in the form [Chunk X].

INSTRUCTIONS FOR YOUR BEHAVIOR:
1. Provide short, precise, and domain-accurate answers.
2. Use ONLY the information stated in the chunks.
3. If the chunks do not contain enough information to answer the query, say:
   "Final Answer: Not enough information is given."
4. List the specific passages that support your conclusion.
5. Do NOT invent, expand, or bring external knowledge.

OUTPUT FORMAT:

Passage: [Chunk X] Exact phrase copied from a chunk.

Passage: [Chunk Y] Another phrase copied from a chunk.
...

Final Answer:
Your answer here.

EXAMPLE:

INPUT: 

Chunk

Invocamos ahora al extractor para estructurar la respuesta del generador. Este obtiene un objeto de tipo `Answer` con dos atributos:

* `annotated_texts:` Una lista de frases extraidas de los chunks de contexto. Cada frase está anotada con el índice del chunk del cual se extrajo. 
* `final_answer`: La respuesta final a la consulta del usuario y fundamentada en las frases anotadas.

Imprimimos tanto la respuesta original del modelo junto a su prompt de sistema. El prompt de usuario del extractor es simplemente el texto en crudo obtenido del generador. 

In [3]:
extractor_invocation, answer = extractor.extract(generator_invocation.raw_answer)

print("<LLM Answer>")
print(extractor_invocation.raw_answer)
print("="*80)
print("<System Prompt>")
print(extractor_invocation.system_prompt)

<LLM Answer>
{"passages": [{"text": "Aristoteles was a philosopher who argued that happiness is achieved by practicing and acquiring virtues.", "chunk_number": 2}], "final_answer": "Happiness is achieved by practicing and acquiring virtues."}

<System Prompt>
Given a text with a list of chunk passages, each one annotated with a number, and a final answer, extract it's content as a JSON file with the following schema:
{'$defs': {'AnnotatedPassage': {'properties': {'text': {'title': 'Text', 'type': 'string'}, 'chunk_number': {'title': 'Chunk Number', 'type': 'integer'}}, 'required': ['text', 'chunk_number'], 'title': 'AnnotatedPassage', 'type': 'object'}}, 'properties': {'passages': {'items': {'$ref': '#/$defs/AnnotatedPassage'}, 'title': 'Passages', 'type': 'array'}, 'final_answer': {'title': 'Final Answer', 'type': 'string'}}, 'required': ['passages', 'final_answer'], 'title': 'Answer', 'type': 'object'}



Imprimimos a continuación la salida en un formato estructurado:

In [4]:
print(f"Answer: {answer.final_answer}")
print("Evidence:")
for passage in answer.passages:
    print(f"[Chunk {passage.chunk_number}] {passage.text}")

Answer: Happiness can be achieved by practicing and acquiring virtues, as argued by Aristoteles.
Evidence:
[Chunk 0] Achieving it requires practicing virtues and learning to recognize both personal and collective emotions.
[Chunk 2] Aristoteles was a philosopher who argued that happiness is achieved by practicing and acquiring virtues.


Excelente trabajo! Ahora invocamos el flujo RAG completo y visualizamos la respuesta del modelo, los chunks recuperados y las frases extraidas en un panel HTML. 

In [7]:
from chunk_qa_panel import render_qa_result

query = input('Enter Query: ')

result = chunk_qa.answer_query(query, k = 3)
render_qa_result(result)

Enter Query:  What are the different parts of the brain?


## 2. Concept Map RAG

El módulo `cmap_qa` contiene las herramientas necesarias para implementar un sistema RAG basado en mapas conceptuales. En este esquema, los objetos semánticos recuperados son los mapas conceptuales y sus ternas de conocimiento, en vez de los documentos escritos en lenguaje natural del RAG clásico. La lógica principal es realizada por la clase `CMapQALogic`, la cual se encarga del control de errores y la sincronización entre el módulo de persistencia y los objetos que interactúan con LLMs. Por defecto, se le solicita al LLM a que genere una respuesta en función de las ternas de conocimiento de los mapas conceptuales recuperados. Las preguntas de enfoque sirven para brindarles un contexto a estas ternas y ayudan a dilucidar sus significados. La búsqueda semántica de los mapas conceptuales es posible porque al darlos de alta, el embedding de cada uno se iguala al embedding de su pregunta de enfoque, aunque otros esquemas de codificación son posibles. 

El método principal es `answer_query`, el cual recibe una consulta escrita en lenguaje natural *query* y el número de mapas conceptuales a recuperar $k$. Su descripción paso a paso es:

1. **Validación:** Se controla que la consulta *query* y el número de chunks a recuperar $k$ sean valores correctos.
2. **Codificación:** Se calcula el embedding de la consulta *query*.
3. **Recuperación:** Se invoca al módulo de persistencia para realizar una búsqueda semántica de los $k$ mapas conceptuales más relevantes a *query*.
4. **Generación:** Se invoca a un objeto del tipo `Generator` y que interactúa con un LLM para responder *query* en función de los mapas conceptuales recuperados. Este genera un objeto del tipo `LLMInvocationData` con la respuesta en crudo del modelo. 
5. **Extracción:** Se invoca a un objeto del tipo `Generator` para analizar sintácticamente la respuesta del generador y estructurar su contenido como un objeto del tipo `Answer`.

El resultado de todo este proceso es un objeto del tipo `CMapQAResult` y que contiene los resultados de la validación, los mapas conceptuales recuperados de la base semántica, las invocaciones del generador y el extractor y una respuesta estructurada a la consulta. Esta última consiste de una lista de ternas de conocimiento extraidas de los mapas conceptuales y que son relevantes a la pregunta de entrada junto a una conclusión redactada a partir de estas ternas.

Importamos a continuación las librerías necesarias, establecemos la conexión con la base de conocimiento y creamos los objetos que nos permiten implementar el sistema RAG. 

In [2]:
from llms_kgs.persistence import CMapRetriever

from llms_kgs.logic.cmap_qa import (
    # Prompts:
    GENERATOR_ZERO_SHOT_SYSTEM_PROMPT,
    GENERATOR_USER_TEMPLATE,

    # Generator:
    PromptFormatter,
    Generator,

    # Extractor:
    Answer,
    Extractor,

    # Logic:
    CMapQALogic,
    CMapQAResult)

from llms_kgs.domain import CMap, Triple

# Instantiate cmap retriever:
retriever = CMapRetriever(connection_provider)

# Instantiate cmap extractor:
cmap_extractor_llm = Gemma3_4B(schema=Answer.model_json_schema())

# Instantiate user prompt formatter:
formatter = PromptFormatter(GENERATOR_USER_TEMPLATE)

# Instantiate generator:
generator = Generator(formatter = formatter, llm = Gemma3_4B(), 
                      system_prompt = GENERATOR_ZERO_SHOT_SYSTEM_PROMPT)

# Instantiate extractor:
extractor = Extractor(llm=cmap_extractor_llm)

# Instantiate main logic: 
cmap_qa = CMapQALogic(retriever=retriever,extractor=extractor,generator=generator,ef=m3)

Prabamos a continuación el funcionamiento del generador sobre un dominio de juguete. Imprimimos la respuesta final del modelo y los prompts de sistema y usuario utilizados.

In [3]:
cmaps = [
CMap(
    focus_question = 'what is anorexia?',
    triples = [
        Triple('anorexia', 'is defined as', 'absence or loss of appetite for food'),
        Triple('anorexia', 'may be a', 'psychological disorder'),
        Triple('anorexia', 'may have', 'physiological causes')]),

CMap(
    focus_question = 'what is range of motion?',
    triples = [
        Triple('range of motion', 'is defined as', 'degree of movement of a joint'),
        Triple('range of motion', 'is determined by', 'contour of the joint'), 
        Triple('range of motion', 'is determined by', 'restraining bones'),
        Triple('range of motion', 'is determined by', 'ligaments of the capsule surrounding the joint')]),

CMap(
    focus_question = 'what is anorexia nervosa?',
    triples = [
        Triple('anorexia nervosa', 'is an', 'eating disorder'),
        Triple('anorexia nervosa', 'occurs frequently in', 'adolescent girls'),
        Triple('anorexia nervosa', 'involves', 'persistent refusal of food'),
        Triple('anorexia nervosa', 'involves', 'excessive fear of weight gain'),
        Triple('anorexia nervosa', 'involves', 'refusal to maintain minimally normal body weight'),
        Triple('anorexia nervosa', 'involves', 'disturbed perception of body image'),
        Triple('anorexia nervosa', 'involves', 'amenorrhea'),
        Triple('amenorrhea', 'is defined as', 'absence of at least three menstrual periods')])
]


generator_invocation = generator.generate(Query('what is amenorrhea?'), cmaps)

print("<LLM Answer>")
print(generator_invocation.raw_answer)
print("="*80)
print("<System Prompt>")
print(generator_invocation.system_prompt)
print("="*80)
print("<User Prompt>")
print(generator_invocation.user_prompt)

<LLM Answer>
Triples:
amenorrhea @ is defined as @ absence of at least three menstrual periods

Final Answer:
Amenorrhea is defined as the absence of at least three menstrual periods.
<System Prompt>
You are an expert in psychology. Your task is to answer user queries ONLY using the information provided in the concept maps.

A concept map consists of:
- A focus question.
- A set of knowledge triples with this format: source @ relation @ target

INSTRUCTIONS FOR YOUR BEHAVIOR:
1. Provide short, precise, and domain-accurate answers.
2. Use ONLY the knowledge explicitly stated in the concept maps.
3. List the specific triples that support your conclusion. 
4. Do NOT invent, expand, or bring external knowledge.
5. If the concept maps do not contain enough information to answer the query, say:
"
Triples:

Final Answer:
Not enough information is given to answer the question.
"

OUTPUT FORMAT:

Triples:
source @ relation @ target
source @ relation @ target
... 

Final Answer:
Your answer here

Invocamos ahora al extractor para estructurar la respuesta del generador. Este obtiene un objeto de tipo `Answer` con dos atributos:

* `triples:` Una lista de ternas de conocimiento extraidas de los mapas conceptuales.
* `final_answer`: La respuesta final a la consulta del usuario y fundamentada en las frases anotadas.

Imprimimos tanto la respuesta original del modelo junto a su prompt de sistema. El prompt de usuario del extractor es simplemente el texto en crudo obtenido del generador. 

In [5]:
extractor_invocation, answer = extractor.extract(generator_invocation.raw_answer)

print("<LLM Answer>")
print(extractor_invocation.raw_answer)
print("="*80)
print("<System Prompt>")
print(extractor_invocation.system_prompt)

<LLM Answer>
{"triples": [{"source": "amenorrhea", "relation": "is defined as", "target": "absence of at least three menstrual periods"}], "final_answer": "Amenorrhea is defined as the absence of at least three menstrual periods."}

<System Prompt>
Given a text with a list of knowledge triples and a final answer, extract it's content as a JSON file with the following schema:
{'$defs': {'Triple': {'properties': {'source': {'title': 'Source', 'type': 'string'}, 'relation': {'title': 'Relation', 'type': 'string'}, 'target': {'title': 'Target', 'type': 'string'}}, 'required': ['source', 'relation', 'target'], 'title': 'Triple', 'type': 'object'}}, 'properties': {'triples': {'items': {'$ref': '#/$defs/Triple'}, 'title': 'Triples', 'type': 'array'}, 'final_answer': {'title': 'Final Answer', 'type': 'string'}}, 'required': ['triples', 'final_answer'], 'title': 'Answer', 'type': 'object'}

Each triple is of the form:
source @ relation @ target



Imprimimos a continuación la salida en un formato estructurado:

In [6]:
print(f"Answer: {answer.final_answer}")
print("Evidence:")
for triple in answer.domain_triples():
    print(f"Triple: {triple.to_sentence()}")

Answer: Amenorrhea is defined as the absence of at least three menstrual periods.
Evidence:
Triple: amenorrhea is defined as absence of at least three menstrual periods


Excelente trabajo! Ahora invocamos el flujo RAG completo y visualizamos la respuesta del modelo, los mapas conceptuales recuperados y las ternas de conocimiento extraidas en un panel HTML. 

In [9]:
from cmap_qa_panel import render_cmap_qa_result

query = input('Enter Query: ')

result = cmap_qa.answer_query(query, k = 3)
render_cmap_qa_result(result)

Enter Query:  What are the main symptoms of depression?


Source,Relation,Target
depression,→ is a →,negative affective state
negative affective state,→ ranges from →,unhappiness and discontent
negative affective state,→ ranges to →,"extreme feeling of sadness, pessimism, and despondency"
negative affective state,→ interferes with →,daily life
depression,→ co-occurs with →,"physical, cognitive, and social changes"
"physical, cognitive, and social changes",→ includes →,altered eating or sleeping habits
"physical, cognitive, and social changes",→ includes →,lack of energy or motivation
"physical, cognitive, and social changes",→ includes →,difficulty concentrating or making decisions
"physical, cognitive, and social changes",→ includes →,withdrawal from social activities
major depressive disorder,→ is a →,mood disorder


## 3. Conclusiones

En esta notebook mostramos las herramientas provistas por nuestra librería para implementar y diagnosticar sistemas RAG interpretables basados en chunks y mapas conceptuales. Las extensiones posibles que pueden realizarse a este sistema son muchas, pero nuestra intención es demostrar que es factible aumentar a los LLMs con bases de conocimiento para mejorar la calidad e interpretación de sus respuestas. 

## Referencias

1. *Retrieval-Augmented Generation for Large Language Models: A Survey* de Yunfan Gao, Yun Xiong, et. al.
2. [APA Dictionary of Psychology](https://dictionary.apa.org/) de American Psychological Association. 